# snapjudge training on Colab

1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Run top to bottom
3. Output lands in `out/`; figures are PNG+SVG in `out/figures`

**Note:** the trainer is C++/BLAS (CPU), not GPU-computed. The T4 selection just gets a solid Colab box; the GPU itself isn't used by the training loop.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
print('cpu threads:', os.cpu_count())


## Dependencies


In [ ]:
!apt-get update -qq && apt-get install -y -qq libpcre2-dev libcurl4-openssl-dev libopenblas-dev cmake git >/dev/null
!cmake --version | head -1


## Clone snapjudge


In [ ]:
!git clone --depth 1 https://github.com/Cyrax321/snapjudge.git
%cd snapjudge
!ls


## Build (CPU path, OpenBLAS on Colab)


In [ ]:
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build build -j $(nproc)
!ls build/snapjudge*


## Sanity: all suites green


In [ ]:
!ctest --test-dir build --output-on-failure


## Train (full epoch, 1,200 train states)


In [ ]:
# Pre-download the RAW base checkpoint (~800MB).
# IMPORTANT: use 'convaiinnovations/laya' (the zero-shot base), NOT
# 'laya-typed-decisions' — that one is already fine-tuned on our data/
# (circular: re-training it on the same data gives ~0 gain).
from huggingface_hub import snapshot_download
snapshot_download(repo_id='convaiinnovations/laya',
                  local_dir='/content/base',
                  allow_patterns=['rl_agent_config.json','model.safetensors',
                                  'tokenizer/*','encoder/*'])
print('base ready')


In [ ]:
!./build/snapjudge-train --base /content/base --data data/train.jsonl --val data/val.jsonl --out out/typed-ft --epochs 1


### (Optional) tune the loss for higher soft-accuracy + calibration

Distillation (raised cross-entropy weight) closes the soft-accuracy gap
vs Jev; label smoothing + temperature annealing close the raw-calibration
gap. LoRA (--lora-r) adapts the encoder output for generalization.


In [ ]:
# Example: distill harder + anneal temperature + rank-8 LoRA
# !./build/snapjudge-train --base /content/base --data data/train.jsonl \
#     --val data/val.jsonl --out out/typed-ft --epochs 1 \
#     --w-nll 2.0 --w-sph 0.2 --label-smoothing 0.05 \
#     --temp-start 2.0 --temp-end 1.0 --lora-r 8


## Evaluate


In [ ]:
!./build/snapjudge-eval --ckpt out/typed-ft --data data/val.jsonl --json out/report.json
!python3 -c "import json; r=json.load(open('out/report.json')); print({k:v for k,v in r.items() if k!='points'})"


## Figures


In [ ]:
!./build/snapjudge-plots --report out/report.json --out out/figures
!ls out/figures


## Look at the figures


In [ ]:
from IPython.display import Image, display
for f in ['calibration','roc','workflows','confusion-choice','confusion-score','confusion-noul','score-spread','pr-noul']:
    p = f'out/figures/{f}.png'
    try:
        display(Image(filename=p))
        print(p)
    except FileNotFoundError:
        pass


## Benchmark vs Laya / Jev

Convert the reference benchmark datasets to typed-decisions JSONL, then
print a head-to-head table against the published Laya and TypeSafe Jev
numbers. One cell, end to end. `--shortlist-k 20` uses the coarse-to-fine
decode path so Banking77's 77 options don't overflow the head budget.


In [ ]:
%%time
import os, json

# 1) convert the four reference benchmarks to typed-decisions JSONL
os.makedirs('benchmarks', exist_ok=True)
!python3 python/export_classification.py \
    --dataset fancyzhx/ag_news --label text --labels 'World,Sports,Business,Sci/Tech' \
    --question "Which news category does this article belong to?" \
    --out benchmarks/ag_news --max-per-split 2000
!python3 python/export_classification.py \
    --dataset dair-ai/emotion --label text --labels 'sadness,joy,love,anger,fear,surprise' \
    --question "What emotion does this text express?" \
    --out benchmarks/emotion --max-per-split 2000
!python3 python/export_classification.py \
    --dataset SetFit/sst5 --label text \
    --labels 'very negative,negative,neutral,positive,very positive' \
    --qtype score --question "What is the sentiment of this sentence?" \
    --out benchmarks/sst5 --max-per-split 2000

# Banking77 labels are derived from its (label -> label_text) mapping
from datasets import load_dataset
d = load_dataset('mteb/banking77')
m = {x['label']: x['label_text'] for x in d['train']}
bank_labels = ','.join(m[i] for i in sorted(m))
!python3 python/export_classification.py \
    --dataset mteb/banking77 --label text --labels "$bank_labels" \
    --question "Which banking intent does this query express?" \
    --out benchmarks/banking77 --max-per-split 2000

# 2) map benchmark name -> val JSONL and run the harness
datasets = {
    'ag_news':      'benchmarks/ag_news/val.jsonl',
    'dair-emotion': 'benchmarks/emotion/val.jsonl',
    'banking77':    'benchmarks/banking77/val.jsonl',
    'sst5':         'benchmarks/sst5/val.jsonl',
}
json.dump(datasets, open('benchmarks/benchmarks.json', 'w'))

!python3 python/run_benchmark.py --ckpt out/typed-ft \
    --datasets benchmarks/benchmarks.json \
    --eval-bin build/snapjudge-eval \
    --shortlist-k 20 --out out/benchmark_report.md


## Download / publish


In [ ]:
!cd out && zip -qr /content/snapjudge-out.zip typed-ft figures report.json
from google.colab import files
files.download('/content/snapjudge-out.zip')


## (Optional) Push the checkpoint to the HF hub


In [ ]:
import os
# os.environ['HF_TOKEN'] = 'hf_...'
import subprocess
r = subprocess.run(['./build/snapjudge-train-push','--ckpt','out/typed-ft','--repo','you/snapjudge-typed-ft'], capture_output=True, text=True, env=os.environ)
print(r.stdout)
print(r.stderr)
print('rc', r.returncode)
